# Marmousi2 Acoustic bv1.2 Inversion Validation

This notebook validates the inversion side of the Marmousi2 acoustic case using synthetic-true observations. Run the forward-modeling notebook first.

## Scope

- `inversion10`: short 3-shot, 10-iteration NPU validation.
- `inversion100`: longer 3-shot, 100-iteration sanity check.

The 100-iteration stage should only be run after inspecting the 10-iteration outputs.

In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "ADFWI").exists():
    REPO_ROOT = Path("/liufeng1afs/project/04_Inversion/ADFWI-github")

SCRIPT = REPO_ROOT / "examples" / "validation" / "marmousi2_acoustic_bv12" / "scripts" / "run_validation.py"
OUTPUT_ROOT = REPO_ROOT / "examples" / "validation" / "marmousi2_acoustic_bv12" / "outputs"
SCRIPT


In [ ]:
def run_stage(stage: str, *, dry_run: bool = True, overwrite: bool = False, device: str = "npu:0", extra_args: list[str] | None = None):
    command = [sys.executable, str(SCRIPT), stage, "--device", device, "--output-root", str(OUTPUT_ROOT)]
    if dry_run:
        command.append("--dry-run")
    if overwrite:
        command.append("--overwrite")
    if extra_args:
        command.extend(extra_args)
    proc = subprocess.run(command, cwd=str(REPO_ROOT), text=True, capture_output=True, check=False)
    if proc.stderr:
        print(proc.stderr)
    if proc.stdout:
        print(proc.stdout)
    if proc.returncode != 0:
        raise RuntimeError(f"stage {stage} failed with return code {proc.returncode}")
    return json.loads(proc.stdout[proc.stdout.find("{"):])


## Preview Inversion Commands

This dry run prints the 10-iteration inversion command without running inversion.

In [ ]:
plan = run_stage("inversion10", dry_run=True, device="npu:0")
plan["stages"][0]["command_text"]

## Run 10-Iteration Inversion

Set `RUN_INVERSION_10 = True` to execute the short inversion validation.

In [ ]:
RUN_INVERSION_10 = False

if RUN_INVERSION_10:
    inversion10_result = run_stage("inversion10", dry_run=False, overwrite=True, device="npu:0")
else:
    inversion10_result = {"status": "skipped", "reason": "set RUN_INVERSION_10=True"}

inversion10_result

## Optional 100-Iteration Inversion

Run this only after the 10-iteration loss curve and model update are reasonable.

In [ ]:
RUN_INVERSION_100 = False

if RUN_INVERSION_100:
    inversion100_result = run_stage("inversion100", dry_run=False, overwrite=True, device="npu:0")
else:
    inversion100_result = {"status": "skipped", "reason": "set RUN_INVERSION_100=True"}

inversion100_result

## Inspect Outputs

Inversion outputs are written under `outputs/inversion10_*` and `outputs/inversion100_*`. Inspect `summary.json`, `loss_history.csv`, `loss_curve.png`, and `vp_initial_final_delta.png`.